# Phase 2: Data Cleaning (ETL)

This notebook loads the raw Yelp Open Dataset JSON files, filters them down to a
single-city working subset, cleans the structured fields, and writes two output
files that later phases will build on:

- `data/business_clean.csv` — one row per business
- `data/reviews_clean.csv` — one row per review, restricted to businesses in
  the business subset

We work with one city (chosen below, based on volume) rather than the full
national dataset, mainly so the review file — which is several GB — stays
manageable in memory once filtered.

## Load business data

The business file is small enough (~120 MB) to load fully into memory in one
shot — no chunking needed here (that's reserved for the much larger review
file below).

In [1]:
import pandas as pd

business = pd.read_json('../data/Yelp JSON/yelp_academic_dataset_business.json', lines=True)
business.shape

(150346, 14)

## Explore the raw structure

Before cleaning anything, check what columns we have, their types, and where
the nulls are. This tells us which columns are safe to keep as-is, which need
cleaning, and which aren't worth the effort for this project's scope.

In [2]:
print(business.dtypes)
print()
print(business.isnull().sum())

business_id      object
name             object
address          object
city             object
state            object
postal_code      object
latitude        float64
longitude       float64
stars           float64
review_count      int64
is_open           int64
attributes       object
categories       object
hours            object
dtype: object

business_id         0
name                0
address             0
city                0
state               0
postal_code         0
latitude            0
longitude           0
stars               0
review_count        0
is_open             0
attributes      13744
categories        103
hours           23223
dtype: int64


`attributes` and `hours` are nested/inconsistent structures (attribute values
are stored as Python-repr strings, not clean JSON — e.g. `"u'free'"`) and
aren't needed for the credit-risk/segmentation/growth features this project
targets, so we'll drop both rather than spend effort parsing them. `categories`
has very few nulls (103 out of 150,346) — small enough to just drop those rows
later, since category is a useful segmentation feature.

## Choose the working city

The real review file is several GB, so we scope the project down to one city
rather than processing the whole country. Check which cities have the most
businesses — that gives us the richest dataset to work with while keeping
review volume manageable.

In [3]:
business['city'].value_counts().head(15)

city
Philadelphia        14569
Tucson               9250
Tampa                9050
Indianapolis         7540
Nashville            6971
New Orleans          6209
Reno                 5935
Edmonton             5054
Saint Louis          4827
Santa Barbara        3829
Boise                2937
Clearwater           2221
Saint Petersburg     1663
Metairie             1643
Sparks               1624
Name: count, dtype: int64

**Philadelphia** has by far the most businesses (14,569 — over 5,000 more than
the next city, Tucson). We'll use it as the working subset for the rest of the
project.

## Clean the business table

Steps:
1. Filter to Philadelphia
2. Drop `attributes` and `hours` (out of scope, see above)
3. Drop rows with missing `categories`
4. Drop duplicate `business_id` rows (safety check — shouldn't be any, but
   worth verifying rather than assuming)

In [4]:
business_clean = business[business['city'] == 'Philadelphia'].copy()

business_clean = business_clean.drop(columns=['attributes', 'hours'])
business_clean = business_clean.dropna(subset=['categories'])
business_clean = business_clean.drop_duplicates(subset=['business_id'])

business_clean.shape

(14560, 12)

`categories` is currently one comma-separated string per business (e.g.
`"Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries"`). We'll leave it as
a string for now — later phases (feature engineering) can split it into a
list or one-hot encode it as needed, and keeping it as a plain string keeps
this CSV simple to read back in. We'll just strip extra whitespace around
each category for consistency.

## Save the cleaned business table

In [5]:
business_clean['categories'] = business_clean['categories'].apply(
    lambda s: ', '.join(part.strip() for part in s.split(','))
)

business_clean.to_csv('../data/business_clean.csv', index=False)
business_clean.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
15,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,1,"Sushi Bars, Restaurants, Japanese"
19,ROeacJQwBeh05Rqg7F6TCg,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,1,"Korean, Restaurants"
28,QdN72BWoyFypdGJhhI5r7g,Bar One,767 S 9th St,Philadelphia,PA,19147,39.939825,-75.157447,4.0,65,0,"Cocktail Bars, Bars, Italian, Nightlife, Resta..."
31,Mjboz24M9NlBeiOJKLEd_Q,DeSandro on Main,4105 Main St,Philadelphia,PA,19127,40.022466,-75.218314,3.0,41,0,"Pizza, Restaurants, Salad, Soup"


## Load and filter reviews (chunked)

The review file is ~5.3 GB — far too large to load into memory in one shot
(this is called out explicitly in CLAUDE.md). Instead we read it in chunks
with `chunksize`, and on each chunk immediately filter down to only the
reviews belonging to businesses in our Philadelphia subset before moving to
the next chunk. This keeps peak memory usage to roughly the size of one chunk
plus the (much smaller) filtered results accumulated so far, rather than the
full 5.3 GB.

This cell will take a few minutes to run since it has to scan the entire file
once, regardless of chunk size.

In [6]:
philly_business_ids = set(business_clean['business_id'])

review_path = '../data/Yelp JSON/yelp_academic_dataset_review.json'
chunk_iter = pd.read_json(review_path, lines=True, chunksize=100_000)

filtered_chunks = []
for i, chunk in enumerate(chunk_iter):
    matched = chunk[chunk['business_id'].isin(philly_business_ids)]
    if len(matched):
        filtered_chunks.append(matched)
    if (i + 1) % 10 == 0:
        print(f'processed {(i + 1) * 100_000:,} reviews so far...')

reviews_clean = pd.concat(filtered_chunks, ignore_index=True)
reviews_clean.shape

processed 1,000,000 reviews so far...


processed 2,000,000 reviews so far...


processed 3,000,000 reviews so far...


processed 4,000,000 reviews so far...


processed 5,000,000 reviews so far...


processed 6,000,000 reviews so far...


processed 7,000,000 reviews so far...


(967489, 9)

## Clean and save the reviews table

- Drop duplicate `review_id` rows (safety check)
- Parse `date` into an actual datetime column — Phase 5c (growth/trend proxy)
  will need a real time dimension to work with, so it's worth doing the
  parsing once here rather than in every downstream notebook

In [7]:
reviews_clean = reviews_clean.drop_duplicates(subset=['review_id'])
reviews_clean['date'] = pd.to_datetime(reviews_clean['date'])

reviews_clean.to_csv('../data/reviews_clean.csv', index=False)
reviews_clean.shape

(967489, 9)

## Sanity checks

Confirm every review's `business_id` actually exists in `business_clean`
(catches any join/filtering mistakes above), and preview both outputs.

In [8]:
assert reviews_clean['business_id'].isin(philly_business_ids).all(), \
    'found reviews for businesses outside the Philadelphia subset'

print(f'{len(business_clean):,} businesses, {len(reviews_clean):,} reviews')
reviews_clean.head()

14,560 businesses, 967,489 reviews


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
1,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
2,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31
3,8JFGBuHMoiNDyfcxuWNtrA,smOvOajNG0lS4Pq7d8g4JQ,RZtGWDLCAtuipwaZ-UfjmQ,4,0,0,0,Good food--loved the gnocchi with marinara\nth...,2009-10-14 19:57:14
4,oyaMhzBSwfGgemSGuZCdwQ,Dd1jQj7S-BFGqRbApFzCFw,YtSqYv1Q_pOltsVPSx54SA,5,0,0,0,Tremendous service (Big shout out to Douglas) ...,2013-06-24 11:21:25
